# Earliest available Bitcoin candle on Binance

This notebook queries the public Binance Spot API to find the earliest available candle for a trading pair. No API key is required.

The daily interval identifies the earliest available trading date. The one-minute interval provides a more precise timestamp for the beginning of the pair's recorded trading activity.

In [4]:
import json
from datetime import datetime, timezone
from urllib.error import HTTPError, URLError
from urllib.parse import urlencode
from urllib.request import Request, urlopen

BASE_URL = "https://data-api.binance.vision"
SYMBOL = "BTCUSD"
INTERVALS = ("1d", "1m")
REQUEST_TIMEOUT_SECONDS = 15

In [5]:
def milliseconds_to_utc(timestamp_ms: int) -> datetime:
    """Convert a Unix timestamp in milliseconds to a UTC datetime."""
    return datetime.fromtimestamp(timestamp_ms / 1_000, tz=timezone.utc)


def request_json(path: str, parameters: dict[str, object]) -> object:
    """Request JSON data from the public Binance market-data API."""
    query_string = urlencode(parameters)
    request = Request(
        f"{BASE_URL}{path}?{query_string}",
        headers={"User-Agent": "tcc-crypto-research/0.1"},
    )

    try:
        with urlopen(request, timeout=REQUEST_TIMEOUT_SECONDS) as response:
            return json.load(response)
    except HTTPError as error:
        response_body = error.read().decode("utf-8", errors="replace")
        raise RuntimeError(
            f"Binance returned HTTP {error.code}: {response_body}"
        ) from error
    except URLError as error:
        raise RuntimeError(f"Could not reach Binance: {error.reason}") from error


def get_earliest_kline(symbol: str, interval: str) -> dict[str, object]:
    """Return the earliest kline available for a symbol and interval."""
    payload = request_json(
        path="/api/v3/klines",
        parameters={
            "symbol": symbol.upper(),
            "interval": interval,
            "startTime": 0,
            "limit": 1,
        },
    )

    if not isinstance(payload, list) or not payload:
        raise ValueError(
            f"No kline data was returned for {symbol.upper()} at {interval}."
        )

    kline = payload[0]
    if not isinstance(kline, list) or len(kline) < 12:
        raise ValueError(f"Unexpected Binance kline response: {kline!r}")

    return {
        "symbol": symbol.upper(),
        "interval": interval,
        "open_time_utc": milliseconds_to_utc(kline[0]),
        "close_time_utc": milliseconds_to_utc(kline[6]),
        "open": kline[1],
        "high": kline[2],
        "low": kline[3],
        "close": kline[4],
        "volume": kline[5],
        "number_of_trades": kline[8],
    }

In [6]:
earliest_klines = [
    get_earliest_kline(symbol=SYMBOL, interval=interval)
    for interval in INTERVALS
]

for kline in earliest_klines:
    print(f"Symbol: {kline['symbol']}")
    print(f"Interval: {kline['interval']}")
    print(f"Earliest open time (UTC): {kline['open_time_utc'].isoformat()}")
    print(f"Candle close time (UTC): {kline['close_time_utc'].isoformat()}")
    print(f"OHLC: {kline['open']} / {kline['high']} / {kline['low']} / {kline['close']}")
    print(f"Volume: {kline['volume']}")
    print(f"Number of trades: {kline['number_of_trades']}")
    print()

Symbol: BTCUSD
Interval: 1d
Earliest open time (UTC): 2025-12-04T00:00:00+00:00
Candle close time (UTC): 2025-12-04T23:59:59.999000+00:00
OHLC: 90990.23000000 / 99000.00000000 / 90990.23000000 / 92409.95000000
Volume: 0.85860000
Number of trades: 285

Symbol: BTCUSD
Interval: 1m
Earliest open time (UTC): 2025-12-04T08:00:00+00:00
Candle close time (UTC): 2025-12-04T08:00:59.999000+00:00
OHLC: 90990.23000000 / 99000.00000000 / 90990.23000000 / 99000.00000000
Volume: 0.01015000
Number of trades: 2



## Interpretation

The result is specific to the selected symbol and interval. A different Bitcoin pair, such as `BTCBRL`, can have a different starting date. Change `SYMBOL` or add another interval to `INTERVALS` and rerun the notebook when comparing markets.

This query asks Binance for one record beginning at Unix epoch time (`startTime=0`). Because the API returns records chronologically when `startTime` is supplied, the returned record is the earliest kline available through this endpoint for the selected pair and interval.